# 🚀 Recommendation System - Full Training Pipeline

## Overview

This notebook orchestrates the complete recommendation system training pipeline:

1. **Phase 1: Data Preparation** (`1_Data_Preparation_Recommendation`)
   * Build user-item interaction matrix
   * Calculate implicit ratings
   * Create train/test split (sequential by order_number)
   * Generate ground truth
   * Output: `ml_features.train_interactions`, `ml_features.test_ground_truth`

2. **Phase 2: Model Training** (`2_Model_Training_Recommendation`)
   * Train Item-Item Collaborative Filtering (baseline)
   * Train ALS Matrix Factorization (advanced)
   * Build Hybrid Ensemble (best of both)
   * Evaluate all models with Precision@K, Recall@K, NDCG@K
   * Save best model to `gold.user_recommendations`

## Expected Runtime
* **Data Preparation**: 5-10 minutes
* **Model Training**: 15-30 minutes  
* **Total**: ~20-40 minutes

## Quick Start

Run all cells below to execute the complete pipeline.

---

In [0]:
import time

print("="*80)
print("PHASE 1: DATA PREPARATION")
print("="*80 + "\n")

start = time.time()

print("Executing: 1_Data_Preparation_Recommendation\n")
print("This will create:")
print("  - ml_features.user_item_interactions")
print("  - ml_features.train_interactions")
print("  - ml_features.test_ground_truth\n")

try:
    # Run Phase 1 notebook
    result = dbutils.notebook.run(
        "./1_Data_Preparation_Recommendation",
        timeout_seconds=1800  # 30 min timeout
    )
    
    elapsed = time.time() - start
    
    print("\n" + "="*80)
    print("\u2705 PHASE 1 COMPLETE")
    print("="*80)
    print(f"Time elapsed: {elapsed/60:.1f} minutes\n")
    
except Exception as e:
    print(f"\n\u274c PHASE 1 FAILED: {str(e)}")
    raise

In [0]:
print("="*80)
print("PHASE 2: MODEL TRAINING")
print("="*80 + "\n")

start = time.time()

print("Executing: 2_Model_Training_Recommendation\n")
print("This will:")
print("  - Train Item-Item CF (baseline)")
print("  - Train ALS model")
print("  - Build Hybrid ensemble")
print("  - Evaluate and compare all models")
print("  - Save best model to gold.user_recommendations\n")

try:
    # Run Phase 2 notebook
    result = dbutils.notebook.run(
        "./2_Model_Training_Recommendation",
        timeout_seconds=3600  # 60 min timeout
    )
    
    elapsed = time.time() - start
    
    print("\n" + "="*80)
    print("\u2705 PHASE 2 COMPLETE")
    print("="*80)
    print(f"Time elapsed: {elapsed/60:.1f} minutes\n")
    
except Exception as e:
    print(f"\n\u274c PHASE 2 FAILED: {str(e)}")
    raise

## ✅ Pipeline Complete - Verify Results

Check the created tables and recommendations.

In [0]:
print("="*80)
print("PIPELINE COMPLETE - VERIFYING OUTPUTS")
print("="*80 + "\n")

# Check ML feature tables
ml_tables = [
    "big_data.ml_features.user_item_interactions",
    "big_data.ml_features.train_interactions",
    "big_data.ml_features.test_ground_truth"
]

print("📊 ML Feature Tables:\n")
for table in ml_tables:
    try:
        count = spark.table(table).count()
        print(f"  ✅ {table}")
        print(f"     Rows: {count:,}\n")
    except Exception as e:
        print(f"  ❌ {table} - NOT FOUND\n")

# Check production recommendations
print("🎯 Production Recommendations:\n")
try:
    prod_table = "big_data.gold.user_recommendations"
    prod_recs = spark.table(prod_table)
    count = prod_recs.count()
    
    print(f"  ✅ {prod_table}")
    print(f"     Users: {count:,}\n")
    
    # Show sample
    print("  Sample Recommendations:")
    prod_recs.select(
        "user_id",
        F.slice("recommendations", 1, 5).alias("top_5"),
        "model_version"
    ).limit(3).show(truncate=False)
    
except Exception as e:
    print(f"  ❌ Production recommendations table not found: {e}\n")

print("="*80)
print("✅ RECOMMENDATION SYSTEM READY FOR PRODUCTION")
print("="*80)

## 📚 Usage Guide

### Query Recommendations for a User

```sql
SELECT 
  user_id,
  recommendations,
  model_version,
  generated_at
FROM big_data.gold.user_recommendations
WHERE user_id = 12345;
```

### Get Top-5 Recommendations

```python
user_id = 12345

recs = spark.table("big_data.gold.user_recommendations") \
    .filter(f"user_id = {user_id}") \
    .select(F.slice("recommendations", 1, 5).alias("top_5")) \
    .collect()[0].top_5

print(f"Top-5 recommended products for user {user_id}: {recs}")
```

### Join with Product Details

```sql
SELECT 
  r.user_id,
  r.recommendations,
  p.product_name,
  p.department,
  p.aisle
FROM big_data.gold.user_recommendations r
LATERAL VIEW explode(slice(r.recommendations, 1, 5)) AS product_id
JOIN big_data.silver.products_enriched p ON p.product_id = product_id
WHERE r.user_id = 12345;
```

### Batch Export for Email Campaign

```python
# Get recommendations for all active users
campaign_recs = spark.sql("""
    SELECT 
        r.user_id,
        slice(r.recommendations, 1, 5) as top_5_products,
        c.email,
        c.segment
    FROM big_data.gold.user_recommendations r
    JOIN big_data.gold.dt_customer_segmentation c ON c.user_id = r.user_id
    WHERE c.segment IN ('Champions', 'Loyal Customers')
""")

campaign_recs.write.mode("overwrite").saveAsTable("big_data.gold.email_campaign_recs")
```

---

## 🔄 Retraining Schedule

**Recommended:** Monthly retraining

* **Why?** As new orders arrive, user preferences evolve
* **How?** Schedule this notebook to run monthly via Databricks Jobs
* **Monitoring:** Track precision@5 over time - retrain if drops >2%

**To Schedule:**
1. Create Databricks Job
2. Set notebook: `0_Run_Full_Pipeline`
3. Schedule: Monthly (1st of each month, midnight)
4. Cluster: Serverless or dedicated ML cluster
5. Alerts: Notify on failure

---

## 📊 Next Steps

1. **A/B Testing**: Test recommendations in production
2. **Monitoring Dashboard**: Track click-through rates, conversion
3. **Feature Enhancement**: Add seasonal patterns, promotions
4. **Real-time API**: Deploy model serving for low-latency inference
5. **Explainability**: Add "why recommended" explanations